# Entrenamiento ligero de 3DMambaIPF en Google Colab

Este cuaderno ejecuta en Colab lo mismo que `train_mamba.py` de la rama `3dmamba-branch`:

1. `prepare`: corta las escenas LiDAR en ventanas de 100 m y las reparte en train/val. Deja fuera el patch de evaluación de `main.py`.
2. `train --task classifier`: fase 1, poda de outliers → `mamba_classifier.pt`
3. `train --task regressor`: fase 2, desplazamientos → `mamba_regressor.pt`

El código del repositorio se usa **sin modificarlo**: el cuaderno lo clona e importa sus funciones. Solo cambia algunos valores de `config.py` en memoria y la configuración de ejecución.

## Antes de empezar

**1. GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4 basta)*.

**2. Datos en Google Drive** (la carpeta `data/` está en `.gitignore`, así que no viene con el `git clone`). Sube **solo estos 3 archivos** (~420 MB), con los mismos nombres de carpeta que en tu `data/` local:

```
MyDrive/3d-denoising/
└── data/
    ├── Sunset_Crater_Volcano_National_Monument_1M/points.laz    # ~4 MB: patch de evaluación (solo para excluirlo del entrenamiento)
    ├── Northern_California_Coast_40M/points.laz                 # ~170 MB: entrenamiento
    └── Sunset_Crater_Volcano_National_Monument_80M/points.laz   # ~250 MB: entrenamiento
```

No subas los archivos de 400k/1M de las otras escenas (dan muy pocas ventanas de 100 m) ni `San_Isabel_NF_CO_Debris_100M`: queda fuera de esta corrida preliminar (necesita ~10 GB de RAM para prepararse y su densidad, 5,5 pt/m², es la más lejana a la del patch de evaluación, 18 pt/m²).

El cuaderno crea `mamba_windows/` (ventanas ya preparadas) y `checkpoints/` en esa misma carpeta de Drive. Así sobreviven a una desconexión, y en las siguientes sesiones se salta `prepare`.

**3. Configuración preliminar** respecto a los valores por defecto de `config.py`, pensada para una T4 de 16 GB y **~2-3 h en total**:

| Parámetro | `config.py` | Preliminar | Efecto |
|---|---|---|---|
| Escenas | 3 | 2 (40M + 80M) | RAM y tiempo de `prepare` |
| `epochs` | 100 | 40 como máximo, ajustado al presupuesto de tiempo | duración |
| `items_per_epoch` / `val_items` | 500 / 60 | 200 / 40 | duración |
| `patch_size` (puntos por parche) | 1000 | 512 | memoria GPU (EdgeConv ∝ N·k) |
| `samples_per_item` (parches por lote) | 8 | 4 | memoria GPU |
| `num_modules` del regresor | 4 | 2 | memoria y tiempo ×½ |

Presupuesto por defecto: **60 min para el clasificador y 120 min para el regresor**. La prueba corta (paso 7) mide la duración real de una época y, si 40 épocas no caben en el presupuesto, reduce las épocas.

`patch_size` y `num_modules` se guardan dentro del checkpoint, y `main.py` los lee de ahí al hacer inferencia. No hay que tocar `config.py` en local para usar estos modelos.

**4. Sesiones de Colab.** Colab gratuito limita el uso de GPU de forma variable y puede cortar una sesión continua de 3-4 h. Se recomienda dividir el trabajo en dos sesiones:
- **Sesión 1:** pasos 1-8 (instalación, `prepare`, prueba corta y clasificador).
- **Sesión 2:** pasos 1-5, 7 y 9. `prepare` se salta solo porque las ventanas ya están en Drive.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import sys, torch
print("Python", sys.version.split()[0], "| torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "No hay GPU: Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU"

## 2. Dependencias

- No se usan las versiones del README (`torch 2.1.2` + `mamba-ssm 1.1.3`): no hay wheels para el Python de Colab (3.12) y habría que compilar. Se usa el **torch que ya trae Colab** con `mamba-ssm` 2.x y `causal-conv1d`, que publican wheels precompiladas para torch 2.6-2.10 (CUDA 12, Python 3.10-3.13). `models/mamba_ipf/backbone.py` importa `mamba_ssm.modules.mamba_simple.Mamba`, que en 2.x tiene la misma firma y los mismos nombres de parámetros. Por eso los checkpoints siguen siendo compatibles con la implementación de referencia de `mamba_fallback.py`, que se usa en local sin CUDA.
- `--no-build-isolation` hace que el instalador use el torch ya instalado para elegir la wheel. Si no encuentra una, **compila desde fuente** (30-60 min). Por eso la celda se detiene antes si tu versión de torch no tiene wheels.
- `open3d` no hace falta para entrenar y no se instala.

In [ ]:
import re, subprocess, sys, torch

torch_mm = tuple(int(v) for v in re.match(r"(\d+)\.(\d+)", torch.__version__).groups())
WHEEL_TORCH = {(2, 6), (2, 7), (2, 8), (2, 9), (2, 10)}   # versiones con wheels de mamba-ssm 2.3.x / causal-conv1d 1.6+

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("laspy[lazrs]")
if torch_mm not in WHEEL_TORCH:
    raise RuntimeError(
        f"torch {torch.__version__} no tiene wheels precompiladas de mamba-ssm: ejecuta la celda de RESPALDO de abajo, "
        "reinicia la sesión y vuelve a empezar desde la celda 1."
    )
pip_install("--no-build-isolation", "causal-conv1d>=1.4", "mamba-ssm>=2.2")
!pip list 2>/dev/null | grep -iE "^(torch|mamba|causal|laspy|lazrs|numpy) "

**RESPALDO (solo si la celda anterior lo pide).** Instala un torch con wheels de mamba-ssm. Después: *Entorno de ejecución → Reiniciar sesión* y vuelve a empezar desde la celda 1. Quita el `#` para ejecutarla.

In [ ]:
# !pip install -q "torch==2.9.*" --index-url https://download.pytorch.org/whl/cu126

## 3. Google Drive y código del repositorio

El código se clona de GitHub (rama `3dmamba-branch`), **no desde tu equipo**. Colab solo ve lo que esté commiteado y pusheado. Si cambias algo en local (por ejemplo `config.py` o `models/mamba_ipf/`), haz `git push` antes de ejecutar esta celda, o cambia el valor en memoria en el paso 5. El hash que imprime la celda debe coincidir con tu `git log -1` local.

In [ ]:
import os, sys
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/3d-denoising")   # <- cámbialo si usas otra carpeta
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WINDOWS = DRIVE_ROOT / "mamba_windows"
DRIVE_CKPT = DRIVE_ROOT / "checkpoints"
DRIVE_CKPT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/Yellow2392/DL-Point-Cloud-Denoising-Pipeline.git"
BRANCH = "3dmamba-branch"
REPO_DIR = Path("/content/3d-denoising-pipeline")

if not REPO_DIR.exists():
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!git -C {REPO_DIR} log -1 --oneline

os.chdir(REPO_DIR)                       # config.py usa rutas relativas (data/, checkpoints/, analytics/)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Archivos .laz en Drive:", sorted(str(p.relative_to(DRIVE_DATA)) for p in DRIVE_DATA.glob("*/points.laz")))

## 4. Verificar los kernels CUDA de Mamba

Se compara el bloque `Mamba` de `mamba_ssm` (CUDA) con la implementación de referencia del repo, usando los mismos pesos. Si coinciden, el entrenamiento usa el camino rápido y el checkpoint resultante se podrá cargar en local con la referencia.

In [ ]:
import torch
from mamba_ssm.modules.mamba_simple import Mamba as MambaCUDA
from models.mamba_ipf.mamba_fallback import Mamba as MambaRef

torch.manual_seed(0)
fast = MambaCUDA(d_model=48).cuda().eval()
ref = MambaRef(d_model=48).cuda().eval()
ref.load_state_dict(fast.state_dict())          # mismos nombres y formas de parámetros
x = torch.randn(2, 256, 48, device="cuda")
with torch.no_grad():
    diff = (fast(x) - ref(x)).abs().max().item()
print(f"Diferencia máxima CUDA vs. referencia: {diff:.2e}")
assert diff < 1e-3, "Los kernels de mamba_ssm no coinciden con la referencia: revisa la instalación"
del fast, ref, x

## 5. Configuración del entrenamiento preliminar

Todo se ajusta aquí. Los cambios a `cfg` solo existen en esta sesión, y `config.py` no se modifica.

- **`TRAIN_TERRAINS`**: 40M + 80M. Cada escena conserva la semilla de su posición en `config.MAMBA_TRAIN_TERRAINS`, así que si más adelante añades `San_Isabel_NF_CO_Debris_100M`, las ventanas de las otras dos no cambian.
- **`TIME_BUDGET_MIN`**: minutos de entrenamiento por modelo. Las épocas se ajustan a este presupuesto en el paso 7, con `LIGHT["epochs"]` como máximo.
- **`cfg.MAMBA_TRAIN_COMBINATIONS`**: degradaciones vistas en entrenamiento. Por defecto son las de `NOISE_COMBINATIONS` (ahora `[["gaussian", "global_outliers"]]`). Conviene que incluya lo que vayas a evaluar en `main.py`.
- Si sale *CUDA out of memory*, baja `samples_per_item` a 2 o `patch_size` a 384.

In [ ]:
import argparse
import config as cfg
import train_mamba as tm

TRAIN_TERRAINS = [
    "Northern_California_Coast_40M",
    "Sunset_Crater_Volcano_National_Monument_80M",
    # "San_Isabel_NF_CO_Debris_100M",   # fuera de la corrida preliminar (~10 GB de RAM en prepare)
]
cfg.MAMBA_WINDOWS_DIR = DRIVE_WINDOWS                   # ventanas en Drive: se preparan una sola vez
cfg.MAMBA_WINDOWS_PER_SPLIT = {"train": 40, "val": 6}  # igual que config.py (las ventanas pesan poco)
# cfg.MAMBA_TRAIN_COMBINATIONS = [["gaussian", "global_outliers"], ["gaussian", "laplacian", "clustered_outliers"]]

LIGHT = dict(
    epochs=40,                # máximo; el paso 7 lo reduce si no cabe en TIME_BUDGET_MIN
    items_per_epoch=200,
    val_items=40,
    patch_size=512,
    samples_per_item=4,
    batch_items=1,
    k_neighbors=cfg.MAMBA_K_NEIGHBORS,
    lr=cfg.MAMBA_LR,
    noise_decay=cfg.MAMBA_NOISE_DECAY,
    num_workers=2,            # Colab gratuito tiene 2 vCPU; la degradación de ventanas se hace en CPU
)
REGRESSOR_MODULES = 2
TIME_BUDGET_MIN = {"classifier": 60, "regressor": 120}
EPOCHS = {task: LIGHT["epochs"] for task in TIME_BUDGET_MIN}   # el paso 7 lo ajusta al presupuesto
PRETRAINED = None            # opcional: Path a un checkpoint del 3DMambaIPF original (PUNet) en Drive

def make_args(task, **overrides):
    # impl="cuda": falla en lugar de caer en silencio en la implementación de referencia (lenta)
    params = dict(LIGHT, device="cuda", impl="cuda", num_modules=REGRESSOR_MODULES, init_from=PRETRAINED, output=None)
    params.update(overrides)
    return argparse.Namespace(task=task, **params)

print("Degradaciones de entrenamiento:", cfg.MAMBA_TRAIN_COMBINATIONS)

## 6. Preparar las ventanas (`prepare`)

Equivale a `python train_mamba.py prepare`, con dos ajustes para Colab:

- **Lectura por bloques del `.laz`.** `load_las_data` mantiene el objeto `laspy` completo en memoria además de las coordenadas. Con escenas de 80-100 M de puntos eso puede superar los ~12 GB de RAM de Colab gratuito. Aquí se leen solo X, Y, Z por bloques: se obtienen los mismos valores en el mismo orden, y por tanto las mismas ventanas.
- **Por escena y en caché.** Las escenas cuyas ventanas ya están en Drive se omiten (pon `FORCE_PREPARE = True` para regenerarlas, por ejemplo si cambias `MAMBA_WINDOWS_PER_SPLIT`). Cada escena conserva la semilla `SEED + i` del script original, de modo que la partición train/val/test es idéntica.

Con 40M + 80M el pico es de unos 7 GB de RAM y tarda ~10-15 min la primera vez. Si la sesión se cae por falta de RAM, ejecuta la celda de nuevo: continúa con las escenas pendientes.

In [ ]:
import gc, shutil
import numpy as np
import laspy
import psutil
from data_loader import load_las_data
from experiment import extract_patch
from models.mamba_ipf.dataset import prepare_windows

FORCE_PREPARE = False

def local_laz(terrain):
    # Copia el .laz de Drive al disco local de Colab (lectura mucho más rápida)
    dst = cfg.DATA_DIR / terrain / "points.laz"
    if not dst.exists():
        src = DRIVE_DATA / terrain / "points.laz"
        if not src.exists():
            raise FileNotFoundError(f"Falta {src}")
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    return dst

def load_xyz(path, chunk_size=5_000_000):
    # Mismo resultado que load_las_data(path)[0], sin retener el resto de atributos
    with laspy.open(path) as reader:
        xyz = np.empty((reader.header.point_count, 3), dtype=np.float64)
        n = 0
        for chunk in reader.chunk_iterator(chunk_size):
            m = len(chunk)
            xyz[n:n + m, 0], xyz[n:n + m, 1], xyz[n:n + m, 2] = chunk.x, chunk.y, chunk.z
            n += m
    return xyz[:n]

def has_windows(terrain):
    return any((DRIVE_WINDOWS / "train").glob(f"{terrain}_*.npz"))

pending = [t for t in TRAIN_TERRAINS if FORCE_PREPARE or not has_windows(t)]
if pending:
    # Caja XY del patch de evaluación de main.py (igual que en train_mamba.prepare)
    eval_points = load_las_data(local_laz(cfg.TERRAIN))[0]
    _, _, (cx, cy) = extract_patch(eval_points, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAX_ATTEMPTS, cfg.PATCH_SEED)
    half = cfg.PATCH_SIZE / 2
    exclude_box = (cx - half, cy - half, cx + half, cy + half)
    del eval_points
    print(f"Patch de evaluación excluido: {exclude_box}")

    for terrain in pending:
        i = cfg.MAMBA_TRAIN_TERRAINS.index(terrain) if terrain in cfg.MAMBA_TRAIN_TERRAINS else len(cfg.MAMBA_TRAIN_TERRAINS)
        for old in DRIVE_WINDOWS.glob(f"*/{terrain}_*.npz"):
            old.unlink()
        points = load_xyz(local_laz(terrain))
        print(f"\n[{terrain}] {len(points):,} puntos | RAM usada: {psutil.virtual_memory().percent:.0f}%")
        saved, n_test = prepare_windows(
            terrain, points, DRIVE_WINDOWS, cfg.PATCH_SIZE, cfg.MIN_POINTS, cfg.MAMBA_WINDOWS_PER_SPLIT,
            cfg.MAMBA_VAL_FRACTION, cfg.MAMBA_TEST_FRACTION, exclude_box, seed=cfg.SEED + i,
        )
        print(f"[{terrain}] ventanas: train={saved['train']}, val={saved['val']} | reservadas para test: {n_test}")
        del points
        gc.collect()

for split in ("train", "val"):
    files = sorted((DRIVE_WINDOWS / split).glob("*.npz"))
    scenes = sorted({f.stem.rsplit("_", 2)[0] for f in files})
    print(f"{split}: {len(files)} ventanas de {scenes}")
extra = {f.stem.rsplit("_", 2)[0] for f in DRIVE_WINDOWS.glob("*/*.npz")} - set(TRAIN_TERRAINS)
if extra:
    print("AVISO: hay ventanas de escenas fuera de TRAIN_TERRAINS y también se usarán:", sorted(extra))

## 7. Prueba corta: memoria, tiempo y épocas que caben en el presupuesto

Dos épocas cortas de cada modelo (no se guardan en Drive). La primera incluye el calentamiento de CUDA, así que se usa la segunda (columna `seconds` del registro) para extrapolar la duración de una época completa. Con eso se fija `EPOCHS[task]` para que el entrenamiento quepa en `TIME_BUDGET_MIN`.

La estimación es algo conservadora: la validación (sin gradientes) cuesta menos que el entrenamiento, pero aquí se cuenta igual. Si sale *CUDA out of memory*, baja `samples_per_item` o `patch_size` en el paso 5 y repite.

Ejecútala en cada sesión antes de entrenar: el resultado depende de la GPU que te asigne Colab.

In [ ]:
import pandas as pd
SMOKE_DIR = Path("/content/smoke")
SMOKE_ITEMS, SMOKE_VAL = 20, 5
full_epoch_items = LIGHT["items_per_epoch"] + LIGHT["val_items"]

for task in ("classifier", "regressor"):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    out = SMOKE_DIR / f"{task}.pt"
    tm.train(make_args(task, epochs=2, items_per_epoch=SMOKE_ITEMS, val_items=SMOKE_VAL, output=out))
    seconds = pd.read_csv(out.with_suffix(".csv"))["seconds"].iloc[-1]
    epoch_min = seconds / (SMOKE_ITEMS + SMOKE_VAL) * full_epoch_items / 60
    EPOCHS[task] = max(1, min(LIGHT["epochs"], int(TIME_BUDGET_MIN[task] // epoch_min)))
    print(f"==> {task}: pico de memoria GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB | "
          f"~{epoch_min:.1f} min/época | {EPOCHS[task]} épocas ≈ {EPOCHS[task] * epoch_min:.0f} min "
          f"(presupuesto {TIME_BUDGET_MIN[task]} min)\n")
print("Épocas a usar:", EPOCHS)

## 8. Entrenar el clasificador (fase 1)

Cada vez que mejora el F1 de validación, el checkpoint se guarda en Drive (`checkpoints/mamba_classifier.pt`), y el registro de cada época queda en `mamba_classifier.csv`.

⚠️ `train_mamba.py` no reanuda: si la sesión se cae, se pierde la época en curso y el mejor checkpoint guardado hasta ese momento sigue en Drive. Al volver a ejecutar la celda se empieza de cero y se sobrescriben el `.pt` y el `.csv`. Si quieres conservarlos, cópialos antes con otro nombre.

Para que Colab no te desconecte por inactividad, mantén la pestaña abierta.

In [ ]:
torch.cuda.empty_cache()
print(f"Clasificador: {EPOCHS['classifier']} épocas")
tm.train(make_args("classifier", epochs=EPOCHS["classifier"], output=DRIVE_CKPT / "mamba_classifier.pt"))

## 9. Entrenar el regresor (fase 2)

Es independiente del clasificador: se puede entrenar en otra sesión o en otro orden. Se guarda cada vez que baja `val_err_after_m` (error medio en metros al punto limpio más cercano después de la red). Compáralo con `val_err_before_m`, el error de la entrada ruidosa.

In [ ]:
torch.cuda.empty_cache()
print(f"Regresor: {EPOCHS['regressor']} épocas")
tm.train(make_args("regressor", epochs=EPOCHS["regressor"], output=DRIVE_CKPT / "mamba_regressor.pt"))

## 10. Curvas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

panels = {
    "mamba_classifier.csv": [("train_loss", "val_loss"), ("val_precision", "val_recall", "val_f1")],
    "mamba_regressor.csv": [("train_loss", "val_loss"), ("val_err_before_m", "val_err_after_m")],
}
for name, groups in panels.items():
    path = DRIVE_CKPT / name
    if not path.exists():
        print(f"(sin {name})")
        continue
    log = pd.read_csv(path)
    fig, axes = plt.subplots(1, len(groups), figsize=(12, 3.5))
    for ax, cols in zip(axes, groups):
        for col in cols:
            ax.plot(log["epoch"], log[col], marker="o", ms=3, label=col)
        ax.set_xlabel("época")
        ax.grid(alpha=0.3)
        ax.legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

## 11. Llevar los checkpoints al equipo local

Los archivos ya están en `MyDrive/3d-denoising/checkpoints/`. Descárgalos desde Drive o con la celda siguiente, y cópialos en `checkpoints/` del repositorio local. Después, `python main.py` ejecutará los pipelines `("mamba", "mamba")`.

En local (sin CUDA) `main.py` usa la implementación de referencia de Mamba: funciona, pero la inferencia es lenta.

In [ ]:
for p in sorted(DRIVE_CKPT.glob("*.pt")):
    ckpt = torch.load(p, map_location="cpu", weights_only=False)
    print(f"{p.name}: kind={ckpt['kind']} | model_args={ckpt['model_args']} | patch_size={ckpt['patch_size']} | "
          f"época={ckpt['extra'].get('epoch')} | {ckpt['extra'].get('monitor')}={ckpt['extra'].get('val', {}).get(ckpt['extra'].get('monitor'))}")

# Descarga directa al navegador (opcional):
# from google.colab import files
# for p in DRIVE_CKPT.glob("mamba_*.*"): files.download(str(p))